<a href="https://colab.research.google.com/github/ovtsxde/poem-generator-rnn/blob/main/poem-generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
with open('pushkin.txt', 'r', ) as f:
  text = f.read()

In [ ]:
print(text[:100])

1823


ПТИЧКА

В чужбине свято наблюдаю
Родной обычай старины:
На волю птичку выпускаю
При светлом п


In [ ]:
import torch
import torch.nn as nn

chars = sorted(list(set(text)))
vocab_size = len(chars)

char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

In [ ]:
class PoemRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=2):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.lstm = nn.LSTM(embedding_dim,
                            hidden_dim,
                            num_layers=num_layers,
                            batch_first=True)

        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, h=None):
        x = self.embedding(x)
        y, h = self.lstm(x, h)

        y = self.dropout(y)
        logit = self.fc(y)
        return logit, h

In [ ]:
EMBEDDING_DIM = 64
HIDDEN_DIM = 128
NUM_LAYERS = 1

device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = PoemRNN(vocab_size=vocab_size,
                embedding_dim=EMBEDDING_DIM,
                hidden_dim=HIDDEN_DIM,
                num_layers=NUM_LAYERS).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0002)

print(model)

PoemRNN(
  (embedding): Embedding(90, 64)
  (lstm): LSTM(64, 128, batch_first=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=128, out_features=90, bias=True)
)


In [ ]:
import numpy as np
from torch.utils.data import Dataset, DataLoader

encoded_text = np.array([char_to_idx[ch] for ch in text], dtype=np.int64)

class TextDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + 1 : idx + self.seq_len + 1]

        return torch.tensor(x), torch.tensor(y)


SEQ_LEN = 100
BATCH_SIZE = 64

dataset = TextDataset(encoded_text, seq_len=SEQ_LEN)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

In [ ]:
def generate_text(model, start_phrase="Я помню ", gen_chars=200, temperature=0.7):
    model.eval()
    input_indices = [char_to_idx[ch] for ch in start_phrase]
    input_tensor = torch.tensor(input_indices, dtype=torch.long).unsqueeze(0).to(device)

    generated_text = start_phrase

    hidden = None

    with torch.inference_mode():
        logits, hidden = model(input_tensor, hidden)

        last_logit = logits[0, -1, :]

        for _ in range(gen_chars):

            scaled_logits = last_logit / temperature

            probabilities = torch.softmax(scaled_logits, dim=-1)

            predicted_idx = torch.multinomial(probabilities, num_samples=1).item()

            predicted_char = idx_to_char[predicted_idx]

            generated_text += predicted_char

            next_input = torch.tensor([[predicted_idx]], dtype=torch.long).to(device)

            logits, hidden = model(next_input, hidden)
            last_logit = logits[0, -1, :]

    return generated_text

In [ ]:
epochs = 10

model.train()

for epoch in range(epochs):
    train_loss = 0
    model.train()
    for batch, (X, y) in enumerate(train_loader):
        X, y = X.to(device), y.to(device)

        logits, _ = model(X)

        loss = loss_fn(logits.view(-1, vocab_size), y.view(-1))
        train_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)

        optimizer.step()

    train_loss /= len(train_loader)
    print(f"Эпоха {epoch} | Средний Loss: {train_loss:.4f}")


    print("\n--- Проверка генерации ---")
    print(generate_text(model, start_phrase="Я помню ", gen_chars=150, temperature=1.2))


Эпоха 0 | Средний Loss: 1.6527

--- Проверка генерации ---
Я помню печанье лстопу.

Не гостули лесненья,
Но чноя стовной мончуюм;
Не серно мит ивой одбадных брустца.

И рахзерь адиднем
Остивниц леменецы!
Слобина сво ж
Эпоха 1 | Средний Loss: 1.6374

--- Проверка генерации ---
Я помню любев был вечамцы твоих.


5 Ла, гин перыво доль? - я ждит выслаженьей:
ЗЗашт лу, дети зановный ххевой:
Мо жлел, ты плучной ребя люблюда,
Кно на я жем
Эпоха 2 | Средний Loss: 1.6220

--- Проверка генерации ---
Я помню вар? узмча; нь чорец младечтал и тог бореего!
Вилпедно блега, гдо в тевда предоже, драновы, гоприжят,
Безсело набедоду бежденьл ожольвих,
Снезбу ситри
Эпоха 3 | Средний Loss: 1.6074

--- Проверка генерации ---
Я помню лене увы;
Визыменьи:
Стегсли, я я, бозду твувать и милою
И в ерань дором, "  И лостеволит склоненьи!

На скла молфнае готоли Нечесью:
И счаста все: дз
Эпоха 4 | Средний Loss: 1.5945

--- Проверка генерации ---
Я помню там убузъ
Полюкорминелты целитастья,
На молго свет, нищою в л

In [ ]:
print(generate_text(model, start_phrase="Делаю это легко ", gen_chars=150, temperature=1.0))

Делаю это легко пыл стурит пламжентестью настариномго говен.


* * *

Налостра: отхара егой:
Темной сенье от третам прозорен!
На тежевая блаженьу,
Отлеять тукины,
Бле
